In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from IPython.display import Markdown
import os

from langchain_community.document_loaders import PyPDFDirectoryLoader
# Path of all Documents : Abount AI Initiative
ai_initiative_pdf_paths = [ f"./resources/{file}" for file in os.listdir("./resources")]

In [ ]:
# Creating PDF loader object
loader = PyPDFDirectoryLoader(path="./resources/")

In [ ]:
from langchain_text_splitters
import RecursiveCharacterTextSplitter
# Split Documents into chunks
text_splitter = RecursiveCharacterTextSplitter
.from_tiktoken_encoder(
encoding_name="cl100k_base",
chunk_size=1000,
chunk_overlap=200
)
# Load and split documents
ai_initiative_chunks = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
### Step 4: Create Vector Store with Embeddings
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
# Initialize OpenAI embedding model
embedding_model = OpenAIEmbeddings(
model="text-embedding-ada-002”
)
# Create vector store from documents
vectorstore = Chroma.from_documents(
documents=ai_initiative_chunks,
embedding=embedding_model,
collection_name="AI_Initiatives"
)
# Create retriever for similarity search
# which fetches 10 relevant chunks
retriever = vectorstore.as_retriever(
search_type="similarity",
search_kwargs={"k": 10}
)

In [ ]:
### Step 5: Test RAG Retrieval (Manual)
# Test query
test_query = "What AI projects is Microsoft working on?"
print(f"🔍 Test Query: {test_query}")
print("="*80)
# Retrieve relevant documents
relevant_docs = retriever.invoke(test_query)
print(f"\n Retrieved {len(relevant_docs)} relevant chunks:\n")
# Display top 3 results
for i, doc in enumerate(relevant_docs[:3], 1):
print(f"📄 Result {i}:")
print(f" Source: {doc.metadata.get('source', 'Unknown')}")
print(f" Page: {doc.metadata.get('page', 'Unknown')}")
print(f" Content: {doc.page_content[:100]}...")
print("-"*80 + "\n")

In [ ]:
@tool
def retriever_tool(query: str) -> str:
""" Interrogez la base de données privée des rapports d’analystes et des documents sur les initiatives en IA.
Cet outil alimenté par RAG effectue des recherches dans des documents internes de l’entreprise concernant les initiatives en IA,
les projets de recherche, les domaines d’innovation et les investissements technologiques stratégiques.
Utilisez cet outil lorsque vous avez besoin d’informations sur :
- Les projets de recherche et les initiatives en IA de l’entreprise
- Les domaines d’innovation en IA et les axes prioritaires
- Les feuilles de route technologiques et les plans futurs
- Les calendriers et les détails de projets d’IA spécifiques
Args:
query: Requête en langage naturel sur les initiatives en IA de l’entreprise
(p. ex., « Sur quels projets d’IA Microsoft travaille-t-elle ? »,
« Les domaines de recherche en IA de NVIDIA »,
« La chronologie de l’IA chez Amazon »)
Returns:
str: Réponse détaillée fondée sur des rapports privés d’analystes avec citations des sources
"""
try:
# Retrieve relevant document chunks
relevant_document_chunks = retriever.invoke(query)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)
return context_for_query
except Exception as e:
return f"Error querying private database: {str(e)}"

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
tools = [retriever_tool]
llm_with_tools = model.bind_tools(tools)

State and System message

In [ ]:
import logging
from typing import Annotated, Sequence, Literal
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain.messages import SystemMessage, HumanMessage, AIMessage

In [ ]:
# Configure logging to see agent's decision-making process
logging.basicConfig(
level=logging.INFO,
format='%(asctime)s [%(levelname)s] %(message)s’,
datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

In [ ]:
class SimpleAgentState(TypedDict):
"""
State for the financial research agent.
Tracks the conversation history with message accumulation.
"""
messages: Annotated[Sequence, add_messages]

In [ ]:
agentic_rag_system_message ="""
Vous êtes un assistant spécialisé dans l’examen des
initiatives en IA des entreprises et dans la fourniture
de réponses exactes fondées sur le contexte fourni.
Les données fornies par l’utilisateur incluront tout le
contexte nécessaire pour répondre à sa question.
Le contexte contient des références à des initiatives,
projets ou programmes d’IA spécifiques d’entreprises,
pertinents pour la requête de l’utilisateur.
Répondez uniquement en utilisant le contexte fourni.
N’ajoutez pas d’informations externes et ne mentionnez
pas le contexte dans votre réponse.
Citez toujours de quelle entreprise provient
l’information.
Si la réponse ne peut pas être trouvée dans le contexte,
répondez par : "Je ne sais pas - cette information n’est
pas disponible dans nos rapports d’analystes."
"""

Agent Node

In [ ]:
def agent_node(state: SimpleAgentState) -> dict:
"""Agent node that calls the LLM with system prompt and current state."""
logger.info("AGENT NODE: Processing request...")
# Prepare messages with system prompt
system_msg = SystemMessage(content=agentic_rag_system_message)
messages = [system_msg] + list(state["messages"])
# Invoke model
logger.info("Calling LLM with tools...")
response = llm_with_tools.invoke(messages)
# Log if agent wants to use tools
if hasattr(response, 'tool_calls') and response.tool_calls:
logger.info(f"✓ Agent decided to use {len(response.tool_calls)} tool(s)")
for i, tool_call in enumerate(response.tool_calls, 1):
logger.info(f" {i}. {tool_call['name']}")
else:
logger.info(" ✓ Agent generated final response (no tools needed)")
return {"messages": [response]}

Condition de routage

In [ ]:
def should_continue(state: SimpleAgentState) -> Literal["tools", "end"]:
"""Determines whether to continue to tools or end."""
last_message = state["messages"][-1]
if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
logger.info("🔀 ROUTING: Continuing to TOOLS node")
return "tools"
logger.info("ROUTING: Ending workflow (final response ready)")
return "end"

creat, compile and display the Graph

In [ ]:
# Create workflow
workflow = StateGraph(SimpleAgentState)
# Create tool node with logging
original_tool_node = ToolNode(tools)
def tool_node_with_logging(state):
logger.info("TOOL NODE: Executing tools...")
result = original_tool_node.invoke(state)
logger.info(f"Tools executed successfully")
return result
# Add nodes to graph
workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node_with_logging)
# Set entry point
workflow.set_entry_point("agent")

In [ ]:
# Add conditional edge from agent
workflow.add_conditional_edges(
"agent",
should_continue,
{
"tools": "tools",
"end": END
}
)
# Add edge from tools back to agent
workflow.add_edge("tools", "agent")
graph = workflow.compile()
logger.info("✅ Enhanced agent created successfully\n")
display(Image(graph.get_graph().draw_mermaid_png()))

Tester l'application Agentic RAG

In [ ]:
print("="*80)
print("TEST 5: Enhanced Agent with RAG - AI Research Activity
Check")
print("="*80 + "\n")
# Test query that requires AI research information
query = "Provide a comprehensive investment analysis for NVIDIA
(NVDA) and Google including their AI research initiatives"
print(f"Query: {query}\n")
print("-"*80 + "\n")
result = graph.invoke(
{"messages": [HumanMessage(content=query)]}
)
print("\n🤖 ENHANCED AGENT RESPONSE (with RAG):")
print("="*80)
print(display(Markdown(result["messages"][-1].content)))
print("\n" + "="*80)
print("\n✅ Notice: The agent now includes:")
print(" • AI research projects from private analyst reports")
print(" • Specific AI initiative details")
print(" • Integration of financial + AI research data")
print(" • Comprehensive investment recommendation")